In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!git clone https://github.com/yanx27/Pointnet_Pointnet2_pytorch.git
%cd Pointnet_Pointnet2_pytorch

In [ ]:
!find . -maxdepth 2 -type f | head -50

In [ ]:
!ls log/sem_seg/pointnet2_sem_seg/checkpoints/

In [ ]:
# The Stanford dataset server is timing out, so we will skip downloading the 4GB dataset.
# Instead, let's load the model directly and feed it synthetic (dummy) data to prove it works
# and understand the EXACT input/output tensor shapes it expects before we use your LiDAR data.

import torch
import sys

sys.path.append('models')
import pointnet2_sem_seg

# 1. Load the exact model architecture for S3DIS (which has 13 classes)
num_classes = 13 
model = pointnet2_sem_seg.get_model(num_classes)

# 2. Load the pretrained weights we found in Cell 4
# Note: weights_only=False is required for older PyTorch checkpoints in PyTorch 2.6+
checkpoint = torch.load('log/sem_seg/pointnet2_sem_seg/checkpoints/best_model.pth', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.cuda() if torch.cuda.is_available() else model
model.eval()
print("✅ Pretrained model and weights loaded successfully!")

# 3. Create dummy point cloud data (Batch, Features, NumPoints)
# The pretrained S3DIS model expects 9 features (usually XYZ, RGB, and normalized spatial coordinates)
B = 1         # Batch size
C = 9         # Features per point
N = 4096      # Number of points in this point cloud slice

dummy_pc = torch.rand(B, C, N)
if torch.cuda.is_available():
    dummy_pc = dummy_pc.cuda()

print(f"\nExpected Input shape to PointNet++: {dummy_pc.shape}")

# 4. Run inference to get predictions!
with torch.no_grad():
    predictions, _ = model(dummy_pc)

print(f"Raw Output shape from PointNet++: {predictions.shape}")

# Convert raw predictions to class labels (0 to 12)
predicted_classes = torch.argmax(predictions, dim=2)
print(f"Final Predicted Classes shape: {predicted_classes.shape}")